# Day 4: RAG & Production Inference
## Retrieval-Augmented Generation · Semantic Search · KV Cache · Batched Inference

**Duration:** ~2 hours | **GPU:** T4 | **Models:** all-MiniLM-L6-v2 (22M) + GPT-2 (124M)

### What you will build

| Step | What happens |
|------|-------------|
| Document corpus | 40 real travel-industry passages (PNR, SSR, GDS, regulations) |
| Embedding index | Sentence embeddings → FAISS vector store |
| Retrieval | Query → cosine similarity → top-k passages |
| Augmented generation | Prepend retrieved context → GPT-2 completion |
| Evaluation | MRR and nDCG on real retrieval results |
| KV cache benchmark | Measure actual speedup with `use_cache=True` vs `False` |
| Batched inference | Sequential vs batched generation — real throughput numbers |

### Why RAG instead of fine-tuning for knowledge injection?

Fine-tuning bakes knowledge into weights — expensive to update and prone to hallucination on facts outside the training window.
RAG keeps knowledge in a **retrieval index** that can be updated in minutes without re-training:

```
Query → [Retriever] → top-k passages → [LLM] → grounded answer
```

The LLM is never asked to memorise facts; it only needs to read and synthesise the retrieved context.

In [ ]:
# Install retrieval + generation stack
!pip install -q sentence-transformers faiss-cpu transformers accelerate

In [ ]:
import torch
import numpy as np
import faiss
import time
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1  Document Corpus — Travel Industry Knowledge Base

The quality of RAG is 80% determined by the quality and coverage of the document corpus.
We use 40 passages across 5 travel-domain topics so the retriever has meaningful distinctions to make.

In [ ]:
# 40 real travel-industry passages across 5 topic clusters
documents = [

    # --- PNR STRUCTURE (8 passages) ---
    "A Passenger Name Record (PNR) is the core booking record in a Global Distribution System. "
    "Every PNR must contain five mandatory elements—remembered as PRINT: Phone contact (P), "
    "Received-from field identifying who made the booking (R), Itinerary with at least one "
    "flight segment (I), passenger Name (N), and Ticketing arrangement or time-limit (T). "
    "The GDS will not allow the End-of-Transaction command until all five are present.",

    "PNR locator codes are six-character alphanumeric identifiers. Each airline and GDS "
    "generates its own locator, so a booking may carry different reference codes in Amadeus "
    "versus the operating carrier's CRS. The locator is used at check-in, for modifications, "
    "and for tracing the record through revenue-accounting and customer-service platforms.",

    "An orphan record occurs when the Received-From or Phone field is missing or malformed. "
    "Downstream reporting systems cannot attribute the transaction to an agent or office, "
    "breaking reconciliation. The fix is to validate all five PRINT elements before allowing "
    "End-of-Transaction, ensuring the general ledger can balance against GDS activity.",

    "Ghost segments are flights with status HX (cancelled by airline) or UC (unable to confirm) "
    "that remain in the PNR without being removed. They confuse the ticketing system and can "
    "prevent ticket issuance. Agencies should run nightly PNR hygiene scripts that scan for "
    "these status codes and issue cancel commands automatically within the 24-hour window "
    "before GDS inactive-segment fees apply.",

    "The Ticketing Time Limit (TKTL) is a datetime field in the PNR set by the airline "
    "indicating when the booking must be ticketed before the seat is released. Agents should "
    "monitor TKTL and trigger payment at least four hours before expiry. Systems can build a "
    "queue that flags any PNR where TKTL is within two hours and ticket status is still Open.",

    "Passive segments (AK/BK codes) are used to record bookings made outside the GDS—such as "
    "a hotel booked directly with the supplier—so the full itinerary appears in one PNR. "
    "Unlike active segments, passive segments do not consume live inventory. They must be "
    "identified separately in analytics to avoid double-counting revenue against direct-channel "
    "bookings and to prevent inflated GDS churn fees.",

    "Re-shopping engines continuously monitor fare prices after a booking is made. If a lower "
    "fare opens up, the net savings formula must account for airline change fees (Category 16) "
    "and agency service fees. Only if the net saving exceeds a threshold should the system "
    "suggest re-booking, otherwise the automation incurs costs rather than saving them.",

    "Back-to-back ticketing is the practice of booking two overlapping trips to circumvent "
    "Saturday-night-stay fare rules. Airlines deploy revenue-integrity robots that detect "
    "PNRs with the same passenger departing on Trip B between the departure and return of "
    "Trip A to the same city pair. Detection results in ticket cancellation and debit memos.",

    # --- SSR / DISABILITY CODES (8 passages) ---
    "Special Service Requests (SSR) are four-letter IATA codes used to communicate passenger "
    "needs to airlines and ground handlers. WCHR means the passenger can walk but needs a "
    "wheelchair for ramp and distance. WCHS means the passenger cannot manage aircraft stairs. "
    "WCHC means the passenger is completely immobile and must be carried to the seat.",

    "For motorised wheelchairs, SSR codes distinguish battery type for hazmat compliance. "
    "WCBD indicates a dry-cell (non-spillable) battery. WCBW indicates a wet-cell (spillable) "
    "battery, which requires additional containment in the cargo hold. Entering WCHR instead "
    "of WCBD causes the ground crew to treat a motorised chair as a manual one, creating "
    "both safety and regulatory risks.",

    "DOT Part 382 (US) mandates that airlines provide wheelchair assistance at no charge, "
    "cannot require advance notice beyond 48 hours for most services, and must return "
    "mobility devices at the aircraft door on arrival. The battery type of a motorised "
    "wheelchair must be documented in the PNR because TSA regulations prohibit certain "
    "lithium battery capacities in the cargo hold without prior approval.",

    "The SSR code mismatch between WCHR and WCBD is a common source of DOT Part 382 "
    "violations. When a Profile-to-PNR sync fails to map the wheelchair battery type, "
    "ground operations receive incorrect instructions. The root cause is usually a missing "
    "field mapping in the profile schema: the Wheelchair_Type attribute exists in the "
    "traveller profile but is not propagated to the SSR segment during PNR creation.",

    "CTCM (Contact Mobile) and CTCE (Contact Email) SSR codes carry passenger contact data "
    "for airline disruption notifications. Airlines parse these with strict regex patterns. "
    "Sending CTCM+1-555-123 when the parser expects CTCM15551234 (no dashes, country code "
    "embedded) causes silent failure. A contact normalisation layer must strip special "
    "characters and validate the country-code prefix before submission.",

    "UMNR (Unaccompanied Minor) triggers the airline's chaperone service. Booking engines "
    "should hard-block self-service for passengers under 12 (Age = TravelDate - BirthDate), "
    "display a call-to-book message, and require escort contact details at origin and "
    "guardian details at destination. These must map to the UMNR SSR code so the cabin "
    "crew briefing includes the correct supervision instructions.",

    "APIS (Advanced Passenger Information System) requires airlines to transmit passport "
    "details—full name, passport number, nationality, expiry date, and destination address—"
    "to government systems before departure. Missing or incorrectly formatted DOCS or DOCO "
    "segments cause the airline API to return Security-Rejected errors. A two-letter country "
    "code versus three-letter mismatch is enough to fail the entire security handshake.",

    "EC 261/2004 entitles EU-bound passengers to cash compensation of up to EUR 600 for "
    "delays over three hours caused by the airline (not extraordinary circumstances). "
    "Automated claim detection requires joining flown-segment data with a flight-status feed "
    "on ArrivalActual - ArrivalScheduled > 180 mins AND origin or carrier is EU. "
    "Extraordinary-circumstances flags must come from a separate tagged data source.",

    # --- GDS CRYPTIC COMMANDS (8 passages) ---
    "Cryptic commands are shorthand instructions sent directly to the GDS airline inventory "
    "system, bypassing the graphical interface. AN (Availability Neutral) retrieves seat "
    "availability; SS (Sell Segment) books a seat; ER or ET issues End-of-Record or "
    "End-of-Transaction to save the PNR. Experienced agents complete complex bookings "
    "40-60 percent faster using cryptic commands than GUI equivalents.",

    "Amadeus and Sabre use different cryptic syntax for the same operations. Amadeus uses "
    "AN for availability; Sabre uses A. Amadeus uses SS1Y1 to sell one Economy seat on "
    "flight 1; Sabre uses 01Y1. Teams migrating between GDS platforms must map command "
    "equivalents carefully, as entering Amadeus syntax in Sabre returns errors that are "
    "not always self-explanatory.",

    "Log-level debugging in GDS environments requires reading raw transaction traces. "
    "Enabling trace mode in Amadeus shows each cryptic command sent, the GDS response "
    "code, and the time taken. Common error codes include NO AVAIL (no seats in requested "
    "class), SEGMENT SELL FAILED (airline rejected the booking), and TTL EXPIRED "
    "(ticketing time limit passed before the transaction was completed).",

    "The ARNK (Arrival Unknown) segment is a placeholder used in the PNR when the "
    "passenger travels between two cities by surface transport—train, car, or ferry. "
    "Without ARNK, the GDS sees a gap in the itinerary and raises an error on pricing "
    "or ticketing. ARNK does not consume inventory; it is a logical connector that keeps "
    "the journey timeline coherent for both the booking system and the traveller's app.",

    "Open-jaw itineraries involve flying into one city and returning from a different city. "
    "In GDS pricing, the system applies Half Round Trip (HRT) combination logic and must "
    "find two fares in the same combinability category (usually Category 10). If the "
    "end-on-end restriction fires, it defaults to two expensive one-way fares. Allowing "
    "multi-city valuation rather than point-to-point surfacing the correct combined price.",

    "Hidden-city ticketing exploits fare construction by booking a cheaper A-B-C itinerary "
    "and disembarking at B. Airlines detect this with differential price checkers that flag "
    "PNRs where Price(A to C via B) is less than Price(A to B). Additional signals include "
    "one-way international bookings with no checked bags. Revenue-integrity teams use these "
    "patterns to prevent short-checking and protect against airline debit memos.",

    "GDS surcharges (coded YQ or as a separate service fee in the fare construction) are "
    "filed by airlines under Category 12 of the ATPCO fare rules. Finance teams often "
    "cannot reconcile these because they are bundled into the Taxes field rather than "
    "labelled separately. Displaying them as Distribution Surcharges and mapping to the "
    "airline's terminology prevents reconciliation disputes.",

    "Point-of-Sale logic means airlines offer different fares based on the GDS Pseudo City "
    "Code (PCC) of the booking office. A UK-based PCC may surface lower base fares in GBP "
    "with different local taxes than a US-based PCC in USD. Analytics systems must always "
    "store the POS country alongside the fare so that currency conversion and local-tax "
    "differences do not distort yield calculations.",

    # --- FARE STRUCTURE (8 passages) ---
    "Booking classes (Y, J, F, Q, K etc.) are inventory buckets within a cabin that carry "
    "different prices and fare rules. Y class is typically the most flexible and expensive "
    "Economy bucket; K is a deeply discounted restricted bucket. The booking class drives "
    "frequent-flyer accrual rates, change and cancellation penalties, and eligibility for "
    "upgrades. Analytics must map single-letter booking codes to their cabin for yield "
    "calculation, as the same letter means different cabins on different airlines.",

    "Inventory nesting means higher booking classes subsume lower ones. If an airline "
    "closes the K bucket, it is shielding that inventory for higher-paying passengers. "
    "Tracking bucket closure velocity—the rate at which low buckets close—is a stronger "
    "demand signal for price prediction than total seats remaining, and is used in revenue "
    "management forecasting models.",

    "ATPCO (Airline Tariff Publishing Company) is the global repository where airlines file "
    "fares and fare rules. Category 16 contains penalty rules (change and cancellation fees). "
    "Category 10 governs combinability. Category 33 covers voluntary refunds. Building "
    "automated refund eligibility tools requires parsing these category tables as structured "
    "data, not keyword-searching the free-text rule output.",

    "Dynamic pricing algorithms adjust fares in real time based on remaining inventory, "
    "competitor pricing, historical booking curves, and demand signals. Price volatility "
    "models trained on GDS data use bucket velocity, days-to-departure, and day-of-week "
    "seasonality as features. The key challenge is separating genuine demand from GDS "
    "churn (repeated cancellation and re-booking of the same seat).",

    "VCN (Virtual Card Numbers) are one-time credit cards generated per hotel booking to "
    "avoid sharing the agency's real card. Reconciliation requires mapping the Virtual Card "
    "ID to the PNR Locator at generation time, then matching the bank transaction ID to "
    "that internal ID when the bank feed arrives. Without this middle link, finance teams "
    "face manual reconciliation of every VCN transaction.",

    "Group bookings operate on a block-inventory model. The agency pays a deposit for a "
    "block of seats at a fixed price, protecting against dynamic pricing jumps. Utilisation "
    "rules require a minimum percentage (typically 80 percent) of seats to be used or the "
    "deposit is forfeited. Group inventory data syncs to the DCS separately from individual "
    "PNR name lists, creating two data streams that must be joined for seat assignment.",

    "Fare construction for circle trips uses Round-the-World or Circle Pacific logic and "
    "triggers the Higher Intermediate Point (HIP) check: if any stop in the circle is "
    "priced higher from the origin than the final destination, the higher price applies to "
    "the whole itinerary. Pricing engines must support sequential valuation across all stops "
    "and calculate taxes at each intermediate city to avoid under-collecting airport fees.",

    "Cabotage laws prohibit foreign airlines from operating domestic routes within a country. "
    "A cabotage filter must check: if Origin Country equals Destination Country AND Carrier "
    "Home Country does not equal Origin Country, block the booking. The exception is "
    "Beyond Rights or Stopover Rights for international itineraries. Search engines that "
    "omit this filter surface illegal domestic flights on foreign carriers.",

    # --- MODERN INFRASTRUCTURE (8 passages) ---
    "IATA Resolution 753 mandates baggage tracking at four scan points: check-in, aircraft "
    "loading, transfer, and arrival. Airlines implement this via SITA's BagMessage (BPM) "
    "stream. Mobile apps showing a baggage progress bar must map raw BPM scan codes to "
    "user-friendly states: Bag Received, Loaded on Flight, In Transit at Hub, Arrived at "
    "Carousel. Null events (missed scans) must default to the last known good position.",

    "Channel management controls hotel inventory distribution across OTAs, GDS, and direct "
    "booking. A property management system (PMS) pushes availability via Direct Inventory "
    "Push (DIP). Push latency—the delay between a PMS update and GDS reflection—causes "
    "Book-to-Fail errors when the GDS shows rooms that the hotel has already sold. "
    "Switching to a pull model for high-demand periods reduces this error rate significantly.",

    "GDPR Article 17 (Right to Erasure) requires deleting PII from traveller profiles on "
    "request, but tax law mandates retaining the financial record of each ticket for seven "
    "years. The solution is data masking: redact Name and Passport fields while retaining "
    "Transaction ID and Amount. The ghost of the sale is preserved for accountants; "
    "the traveller's identity is erased for privacy compliance.",

    "Bilateral Air Service Agreements between countries set the number of flight frequencies "
    "and seat capacities permitted on each route. Revenue forecasting models that ignore "
    "these treaty caps produce impossible predictions. The ICAO database provides "
    "frequency limits by country pair; adding a Regulatory Ceiling variable to "
    "demand forecasts prevents over-prediction on capacity-constrained routes.",

    "The Montreal Convention sets the maximum airline liability for lost or damaged baggage "
    "at 1,288 Special Drawing Rights (SDRs). SDRs are IMF composite currency units that "
    "must be converted to local currency daily. Baggage claim tools must calculate "
    "min(user_declared_value, 1288 * current_SDR_rate) and label the result accurately "
    "so customer-service agents do not over-promise compensation.",

    "CRS-to-GDS synchronisation latency occurs when the airline's Computer Reservations "
    "System (the source of truth) has not yet propagated an inventory change to the "
    "Global Distribution System (the distribution layer). The lag ranges from seconds to "
    "minutes. Booking flows should perform a real-time Sell command against the CRS "
    "before taking payment, rather than relying on cached GDS availability data.",

    "Visa-on-Arrival eligibility logic must check both passport nationality and the "
    "presence of an onward travel segment and confirmed accommodation in the PNR. "
    "Airlines are liable for repatriation costs if they board a passenger who is denied "
    "entry, so VoA eligibility warnings must be shown at search time rather than only "
    "at check-in. Missing return ticket or hotel segment should trigger a mandatory warning.",

    "OFAC sanctions screening requires cross-referencing hotel parent companies and street "
    "addresses against the Specially Designated Nationals list. For US-billing-address "
    "users, sanctioned properties must be hard-hidden in search results, not just shown "
    "as sold-out. Properties owned by sanctioned military groups are an edge case that "
    "requires entity-resolution logic beyond simple name matching.",
]

print(f"Corpus loaded: {len(documents)} passages")
print(f"Avg passage length: {sum(len(d.split()) for d in documents) // len(documents)} words")
print(f"\nSample passages:")
for i in [0, 8, 16, 24, 32]:
    print(f"  [{i:02d}] {documents[i][:80]}...")


## 2  Semantic Embeddings + FAISS Index

Classical keyword search (BM25, TF-IDF) fails on paraphrases — "motorised chair battery" won't match "WCBD dry cell."
**Sentence embeddings** map text into a high-dimensional space where *semantic* similarity = *geometric* proximity.

`all-MiniLM-L6-v2` is a 22M-parameter distilled model trained specifically for semantic similarity.
It maps any text to a 384-dimensional vector in milliseconds.

**FAISS** (Facebook AI Similarity Search) builds an index over these vectors for sub-millisecond nearest-neighbour search —
essential when the corpus grows to millions of documents.

In [ ]:
# Load the embedding model (22M params, 384-dim output)
print("Loading sentence embedding model...")
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

# Embed all documents
print("Embedding document corpus...")
t0 = time.time()
doc_embeddings = embedder.encode(documents, batch_size=32, show_progress_bar=True,
                                  convert_to_numpy=True, normalize_embeddings=True)
embed_time = time.time() - t0

print(f"\nEmbedding complete:")
print(f"  Corpus size   : {len(documents)} passages")
print(f"  Embedding dim : {doc_embeddings.shape[1]}")
print(f"  Time          : {embed_time:.2f}s ({embed_time/len(documents)*1000:.1f} ms/doc)")
print(f"  Index memory  : {doc_embeddings.nbytes / 1e6:.2f} MB")

# Build FAISS flat inner-product index (= cosine similarity after L2 normalisation)
dim = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(doc_embeddings.astype(np.float32))

print(f"\nFAISS index built: {index.ntotal} vectors indexed")

## 3  Retrieval

Given a query, we:
1. Embed the query with the same model
2. L2-normalise (so inner product = cosine similarity)
3. Search FAISS for top-k nearest neighbours
4. Return the matching passages + similarity scores

Cosine similarity of 1.0 = identical; 0.0 = orthogonal (unrelated); negative = opposite meaning.

In [ ]:
def retrieve(query: str, k: int = 3) -> list[tuple[str, float]]:
    query_emb = embedder.encode([query], normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(query_emb, k)
    return [(documents[idx], float(scores[0][rank])) for rank, idx in enumerate(indices[0])]

# --- Test with domain-specific queries ---
test_queries = [
    "What are the mandatory fields needed to complete a PNR booking?",
    "How do I handle a motorised wheelchair with a lithium battery in the PNR?",
    "What is the difference between WCHR and WCBD SSR codes?",
    "Why is my hotel room showing available on the GDS but the hotel says it is sold out?",
    "How does the airline calculate baggage compensation under international treaties?",
]

print("=" * 70)
print("RETRIEVAL RESULTS")
print("=" * 70)
for query in test_queries:
    results = retrieve(query, k=3)
    print(f"\nQuery: {query}")
    for rank, (passage, score) in enumerate(results):
        print(f"  [{rank+1}] score={score:.4f}  {passage[:100]}...")

## 4  Augmented Generation

The RAG pipeline has three stages:

```
1. Retrieve  → top-k passages from FAISS
2. Augment   → prepend passages as context in the prompt
3. Generate  → LLM reads the context and produces an answer
```

We use GPT-2 (124M) here to keep GPU time under 2 minutes.
In production you would swap in an instruction-tuned model (Gemma, Llama, etc.) — the retrieval pipeline is identical.

**Important:** GPT-2 is a completion model, not instruction-tuned, so outputs will be fluent but not perfectly answerable.
The point of this cell is to verify that the retrieved context actually appears in the prompt window.

In [ ]:
print("Loading generation model (GPT-2)...")
gen_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gen_model = AutoModelForCausalLM.from_pretrained("gpt2", torch_dtype=torch.float16).to(device)
gen_tokenizer.pad_token = gen_tokenizer.eos_token
print(f"Model loaded: {sum(p.numel() for p in gen_model.parameters())/1e6:.0f}M parameters")

def rag_generate(question: str, k: int = 3, max_new_tokens: int = 120) -> dict:
    # 1. Retrieve
    retrieved = retrieve(question, k=k)
    context = "\n\n".join([f"[{i+1}] {p}" for i, (p, _) in enumerate(retrieved)])

    # 2. Augment
    prompt = (
        f"Travel industry knowledge:\n{context}\n\n"
        f"Question: {question}\n"
        f"Answer based on the above context:"
    )

    # 3. Generate
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True,
                            max_length=900).to(device)
    prompt_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_k=50,
            repetition_penalty=1.2,
            pad_token_id=gen_tokenizer.eos_token_id,
        )

    answer = gen_tokenizer.decode(output_ids[0][prompt_len:], skip_special_tokens=True)
    return {"question": question, "context_passages": len(retrieved),
            "prompt_tokens": prompt_len, "answer": answer.strip()}

# Run the RAG pipeline
print("\n" + "=" * 70)
print("RAG PIPELINE OUTPUT")
print("=" * 70)
demo_questions = [
    "What mandatory elements does a PNR need to be saved?",
    "What is the SSR code difference between a manual and motorised wheelchair?",
]
for q in demo_questions:
    result = rag_generate(q)
    print(f"\nQ: {result['question']}")
    print(f"Context passages used: {result['context_passages']}")
    print(f"Prompt length: {result['prompt_tokens']} tokens")
    print(f"Answer: {result['answer'][:400]}")

## 5  Retrieval Evaluation — MRR and nDCG

To know whether retrieval is good, we need **labelled relevance**: for each query, which passages *should* be retrieved?

| Metric | What it measures |
|--------|-----------------|
| **MRR** (Mean Reciprocal Rank) | How early does the first relevant passage appear? 1.0 = rank 1, 0.5 = rank 2 |
| **nDCG@k** | Graded relevance: top positions matter more; a perfect ranking scores 1.0 |

We create 5 queries with known-relevant passage indices and evaluate our retriever.

In [ ]:
def reciprocal_rank(retrieved_indices: list, relevant_indices: set) -> float:
    for rank, idx in enumerate(retrieved_indices, start=1):
        if idx in relevant_indices:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(retrieved_indices: list, relevant_indices: set, k: int = 5) -> float:
    import math
    dcg = sum(
        1.0 / math.log2(rank + 1)
        for rank, idx in enumerate(retrieved_indices[:k], start=1)
        if idx in relevant_indices
    )
    # Ideal DCG: all relevant docs at top positions
    ideal = sum(1.0 / math.log2(r + 1) for r in range(1, min(len(relevant_indices), k) + 1))
    return dcg / ideal if ideal > 0 else 0.0

# Labelled evaluation set (query, relevant passage indices)
eval_set = [
    ("mandatory fields in a PNR booking", {0, 1, 2}),
    ("motorised wheelchair battery SSR codes", {8, 9, 10}),
    ("how do cryptic GDS commands work", {16, 17, 18}),
    ("what is inventory nesting in airline booking classes", {25, 26}),
    ("GDPR right to erasure and financial record retention", {34}),
]

print("=" * 70)
print("RETRIEVAL EVALUATION  (k=5)")
print("=" * 70)
print(f"{'Query':<55} {'MRR':>6} {'nDCG@5':>7}")
print("-" * 70)
mrr_scores, ndcg_scores = [], []

for query, relevant_ids in eval_set:
    results = retrieve(query, k=5)
    retrieved_doc_ids = [documents.index(p) for p, _ in results]
    rr  = reciprocal_rank(retrieved_doc_ids, relevant_ids)
    ndcg = ndcg_at_k(retrieved_doc_ids, relevant_ids, k=5)
    mrr_scores.append(rr)
    ndcg_scores.append(ndcg)
    print(f"{query[:54]:<55} {rr:>6.3f} {ndcg:>7.3f}")

print("-" * 70)
print(f"{'Mean':>55} {np.mean(mrr_scores):>6.3f} {np.mean(ndcg_scores):>7.3f}")
print("\nnDCG=1.0 is perfect. MRR=1.0 means relevant passage always retrieved at rank 1.")

## 6  KV Cache Benchmark

During autoregressive generation, each new token attends to *all* previous tokens.
Without caching, every forward pass recomputes Keys and Values for the entire sequence — O(n²) per token.
KV cache stores K and V tensors from previous steps so only the new token needs computation — O(n) per token.

```
Without cache: token t requires attention over [t-1, t-2, ..., 0]  ← full recompute each step
With cache   : token t appends to stored K/V tensors                ← only one new projection
```

The speedup is largest for long sequences. We measure it directly:

In [ ]:
print("=" * 70)
print("KV CACHE BENCHMARK  (GPT-2, real timing)")
print("=" * 70)

# Use a long prompt to make the cache difference measurable
long_context = "\n".join([d[:200] for d in documents[:5]])
prompt = f"Context:\n{long_context}\n\nSummarise the key takeaways:"
inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True,
                        max_length=800).to(device)
prompt_tokens = inputs["input_ids"].shape[1]

N_RUNS = 3
MAX_NEW = 80

print(f"\nPrompt length: {prompt_tokens} tokens | Generating: {MAX_NEW} new tokens | Runs: {N_RUNS}")

# Without KV cache
times_no_cache = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    with torch.no_grad():
        gen_model.generate(**inputs, max_new_tokens=MAX_NEW, do_sample=False, use_cache=False)
    times_no_cache.append(time.perf_counter() - t0)

# With KV cache (default)
times_cache = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    with torch.no_grad():
        gen_model.generate(**inputs, max_new_tokens=MAX_NEW, do_sample=False, use_cache=True)
    times_cache.append(time.perf_counter() - t0)

t_no  = np.mean(times_no_cache)
t_yes = np.mean(times_cache)
speedup = t_no / t_yes
tpt_no  = MAX_NEW / t_no
tpt_yes = MAX_NEW / t_yes

print(f"\n{'Setting':<25} {'Time (s)':>10} {'tok/s':>10}")
print("-" * 47)
print(f"{'Without KV cache':<25} {t_no:>10.2f} {tpt_no:>10.1f}")
print(f"{'With KV cache':<25} {t_yes:>10.2f} {tpt_yes:>10.1f}")
print(f"\nSpeedup: {speedup:.1f}x  |  Throughput gain: {tpt_yes/tpt_no:.1f}x")

## 7  Batched Inference

Running one request at a time leaves GPU utilisation at 10-20%.
Batching multiple sequences lets the GPU parallelise matrix multiplications across requests.

**Tradeoff:** batching requires padding shorter sequences to the length of the longest,
wasting computation on padding tokens. Continuous batching (used in vLLM) solves this by
dynamically grouping requests of similar length — but that requires a serving framework.

Here we measure the simpler static-batch speedup:

In [ ]:
print("=" * 70)
print("BATCHED INFERENCE BENCHMARK")
print("=" * 70)

batch_questions = [
    "What are the mandatory PNR fields?",
    "Explain WCHR vs WCBD SSR codes.",
    "How does KV cache improve inference speed?",
    "What does inventory nesting mean for fare pricing?",
    "When does a ghost segment block ticket issuance?",
    "How is baggage compensation calculated under the Montreal Convention?",
    "What is the difference between active and passive GDS segments?",
    "How does point-of-sale logic affect fare pricing?",
]
MAX_NEW_BATCH = 60

# Sequential
print(f"\nSequential ({len(batch_questions)} requests, {MAX_NEW_BATCH} tokens each)...")
t0 = time.perf_counter()
for q in batch_questions:
    inp = gen_tokenizer(q, return_tensors="pt").to(device)
    with torch.no_grad():
        gen_model.generate(**inp, max_new_tokens=MAX_NEW_BATCH, do_sample=False,
                            pad_token_id=gen_tokenizer.eos_token_id)
seq_time = time.perf_counter() - t0

# Batched
print(f"Batched  ({len(batch_questions)} requests, {MAX_NEW_BATCH} tokens each)...")
gen_tokenizer.padding_side = "left"
batch_inputs = gen_tokenizer(
    batch_questions, return_tensors="pt", padding=True, truncation=True, max_length=200
).to(device)
t0 = time.perf_counter()
with torch.no_grad():
    gen_model.generate(**batch_inputs, max_new_tokens=MAX_NEW_BATCH, do_sample=False,
                        pad_token_id=gen_tokenizer.eos_token_id)
batch_time = time.perf_counter() - t0

seq_tps   = len(batch_questions) * MAX_NEW_BATCH / seq_time
batch_tps = len(batch_questions) * MAX_NEW_BATCH / batch_time

print(f"\n{'Mode':<20} {'Wall time (s)':>14} {'Total tok/s':>12}")
print("-" * 48)
print(f"{'Sequential':<20} {seq_time:>14.2f} {seq_tps:>12.1f}")
print(f"{'Batched':<20} {batch_time:>14.2f} {batch_tps:>12.1f}")
print(f"\nBatch throughput improvement: {batch_tps/seq_tps:.1f}x")
print(f"Note: padding overhead reduces gains; left-padding is used for GPT-2 batching.")

## Day 4 Summary

| Component | What you built | Key metric |
|-----------|---------------|------------|
| Document corpus | 40 real travel-industry passages | 5 topic clusters |
| Embedding model | `all-MiniLM-L6-v2` (22M) | 384-dim, cosine similarity |
| Vector index | FAISS `IndexFlatIP` | Sub-ms search over 40 docs |
| RAG pipeline | Retrieve → Augment → GPT-2 generate | End-to-end working |
| Retrieval eval | MRR + nDCG@5 on labelled queries | Real quality numbers |
| KV cache bench | `use_cache=True` vs `False` | Real measured speedup |
| Batch inference | Sequential vs padded batch | Real throughput ratio |

**Day 5 preview:** Observability — logging latency, token cost, and quality metrics in production; setting up alerting for hallucination rate and retrieval quality drift.